# ZDNABERT: предсказание Z-ДНК у *Eledone cirrhosa* (пункт 3)

Адаптация reference-ноутбука [mitiau/Z-DNABERT](https://github.com/mitiau/Z-DNABERT) под локальную машину (CPU) и геном *Eledone*.

Модель — DNABERT, дообученный на экспериментальных Z-flipon'ах (token classification: вероятность Z-формы на каждый нуклеотид). Своей модели для беспозвоночных нет, берём человеческую (`HG kouzine`) — это перенос, для курсовой допустимо; параметры запуска отметим в отчёте.

**Про OOM (важно):** в оригинале k-меры строятся сразу для всей хромосомы и в памяти копится трек на всю её длину → на больших хромосомах падает Out Of Memory. Здесь хромосома режется на **куски** (`CHUNK`, по умолчанию 200 кб) и обрабатывается по частям — в памяти процесса только его кусок.

**Скорость:** инференс параллелится по кускам через процессы (`N_PROC` × `THREADS_PER`). По бенчмаркам на этой машине задача упирается в пропускную способность памяти: потоки сверх ~24 не помогают, процессы дают максимум ~2.3x. Главный рычаг — **охват `MAX_BP`** (считать кусок хромосомы, а не все 442 Мб).

Результат: `results/bed/zdnabert.bed`.

> Параллель использует модуль `zdna_worker.py` (рядом с проектом) — он нужен для multiprocessing spawn. CPU-only; для пилота ограничь `MAX_BP`.

## 0. Установка и базовая директория

In [1]:
%pip install -q transformers torch biopython gdown scipy numpy

import os
from pathlib import Path

base_dir = Path("/home/bulat_kharisov/inf")
(base_dir / "results" / "bed").mkdir(parents=True, exist_ok=True)
(base_dir / "models").mkdir(parents=True, exist_ok=True)
os.chdir(base_dir)
print("base_dir:", base_dir, "| cwd:", os.getcwd())


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
base_dir: /home/bulat_kharisov/inf | cwd: /home/bulat_kharisov/inf


## 1. Скачать модель ZDNABERT (Google Drive)

Веса и токенизатор лежат на Google Drive, тянем через `gdown` в папку `models/6-new-12w-0/`. Доступные модели: `HG chipseq`, `HG kouzine`, `MM curax`, `MM kouzine`. По умолчанию — `HG kouzine` (как в reference).

In [2]:
import gdown, os, shutil

MODEL_IDS = {
    "HG chipseq": "1VAsp8I904y_J0PUhAQqpSlCn1IqfG0FB",
    "HG kouzine": "1dAeAt5Gu2cadwDhbc7OnenUgDLHlUvkx",
    "MM curax":   "1W6GEgHNoitlB-xXJbLJ_jDW4BF35W1Sd",
    "MM kouzine": "1dXpQFmheClKXIEoqcZ7kgCwx6hzVCv3H",
}
MODEL = "HG kouzine"
MODEL_DIR = base_dir / "models" / "6-new-12w-0"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# общие для всех моделей файлы токенизатора/конфига
AUX = {
    "config.json":              "10sF8Ywktd96HqAL0CwvlZZUUGj05CGk5",
    "special_tokens_map.json":  "16bT7HDv71aRwyh3gBUbKwign1mtyLD2d",
    "tokenizer_config.json":    "1EE9goZ2JRSD8UTx501q71lGCk-CK3kqG",
    "vocab.txt":                "1gZZdtAoDnDiLQqjQfGyuwt268Pe5sXW0",
}

if not (MODEL_DIR / "pytorch_model.bin").exists():
    gdown.download(id=MODEL_IDS[MODEL], output=str(MODEL_DIR / "pytorch_model.bin"), quiet=False)
for fname, fid in AUX.items():
    if not (MODEL_DIR / fname).exists():
        gdown.download(id=fid, output=str(MODEL_DIR / fname), quiet=False)

print("файлы модели:", sorted(p.name for p in MODEL_DIR.iterdir()))

файлы модели: ['config.json', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.txt']


## 2. Загрузка модели и helper-функции

`device` выберется автоматически (CPU, если нет GPU). Функции те же, что в reference: `seq2kmer`, `split_seq` (куски по 512 с перекрытием 16), `stitch_np_seq` (сшивка кусков обратно).

In [3]:
# import torch, numpy as np, scipy.ndimage
# from transformers import BertTokenizer, BertForTokenClassification

# device = "cuda" if torch.cuda.is_available() else "cpu"
# torch.set_num_threads(96)   # все логические ядра; на CPU выигрыш не гарантирован, сравни it/s
# print("torch threads:", torch.get_num_threads())
# tokenizer = BertTokenizer.from_pretrained(str(MODEL_DIR))
# model = BertForTokenClassification.from_pretrained(str(MODEL_DIR)).to(device).eval()
# print("device:", device)

# def seq2kmer(seq, k=6):
#     return [seq[x:x+k] for x in range(len(seq)+1-k)]

# def split_seq(seq, length=512, pad=16):
#     res = []
#     for st in range(0, len(seq), length-pad):
#         res.append(seq[st:min(st+length, len(seq))])
#     return res

# def stitch_np_seq(np_seqs, pad=16):
#     res = np.array([])
#     for s in np_seqs:
#         res = res[:-pad] if len(res) else res
#         res = np.concatenate([res, s])
#     return res

# @torch.no_grad()
# def predict_track(sub_seq, batch_size=64):
#     pieces = split_seq(seq2kmer(sub_seq.upper(), 6))
#     encoded = [tokenizer.encode(" ".join(p), add_special_tokens=False) for p in pieces]
#     preds = []
#     for i in range(0, len(encoded), batch_size):
#         batch = encoded[i:i + batch_size]
#         maxlen = max(len(x) for x in batch)
#         input_ids = torch.zeros(len(batch), maxlen, dtype=torch.long)
#         attn = torch.zeros(len(batch), maxlen, dtype=torch.long)
#         for j, x in enumerate(batch):
#             input_ids[j, :len(x)] = torch.tensor(x, dtype=torch.long)
#             attn[j, :len(x)] = 1
#         input_ids = input_ids.to(device); attn = attn.to(device)
#         probs = torch.softmax(model(input_ids, attention_mask=attn)[-1], dim=-1)[:, :, 1]
#         for j, x in enumerate(batch):
#             preds.append(probs[j, :len(x)].cpu().numpy())   # срез до реальной длины
#     return stitch_np_seq(preds)

## 3. Параметры запуска

- `CHROMS` — какие хромосомы/скаффолды обрабатывать. По умолчанию одна (`"1"`). `None` = все (очень долго).
- `MAX_BP` — сколько bp на хромосому реально считать. **Главный рычаг скорости.** Вся chr1 (442 Мб) ≈ часы; первые 10 Мб — минуты. `None` = вся хромосома.
- `CHUNK` — размер куска-задачи в bp (анти-OOM): геном режется на куски по `CHUNK`, каждый кусок считается отдельным процессом. В памяти процесса — только его кусок.
- `OVERLAP` — перекрытие кусков, чтобы не резать Z-участки на стыках (потом сливаем).
- `N_PROC` × `THREADS_PER` — параллель: процессов и потоков на процесс (держи `N_PROC*THREADS_PER ≤ 96`). По бенчмаркам оптимум ~`8×12`/`12×8` (~2.3x; выше — упор в память).
- `BATCH_SIZE` — размер батча кусков внутри процесса.
- `THRESHOLD`, `MIN_LEN` — порог вероятности и минимальная длина Z-участка (как в reference).
- `CACHE_DIR` — папка per-chunk кэша. Каждый кусок (id = `{chrom}_{start}_{end}`) пишет свой `.tsv`; при перезапуске готовые куски берутся из кэша (модель для них не грузится). В `params.json` хранятся параметры, влияющие на результат (`model`, геном, `CHUNK`, `OVERLAP`, `THRESHOLD`, `MIN_LEN`) — при несовпадении запуск падает, чтобы не смешать разные прогоны. `BATCH_SIZE`/`N_PROC`/`THREADS_PER`/`MAX_BP` на кэш не влияют.

In [4]:
FASTA_IN  = base_dir / "data" / "genome.fa"     # распакованный геном (из тетрадки пунктов 1-2)
BED_OUT   = base_dir / "results" / "bed" / "zdnabert.bed"
CACHE_DIR = base_dir / "results" / "cache" / "zdnabert"   # per-chunk кэш + params.json

CHROMS    = ["1"]          # список имён; None = все хромосомы
MAX_BP    = 2_400_000     # bp на хромосому (пилот); None = вся (chr1 ~ часы!)
CHUNK     = 200_000        # bp на один кусок-задачу
OVERLAP   = 1_000          # перекрытие кусков, чтобы не резать Z-сайты на границе
THRESHOLD = 0.5
MIN_LEN   = 10

# параллель (см. бенчмарки): N_PROC*THREADS_PER <= 96
N_PROC      = 6
THREADS_PER = 16
BATCH_SIZE  = 64

assert FASTA_IN.exists(), f"нет генома {FASTA_IN} — сначала скачай и распакуй unmasked.fa.gz"

In [5]:
BED_OUT

PosixPath('/home/bulat_kharisov/inf/results/bed/zdnabert.bed')

In [6]:
import sys, importlib, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import defaultdict
from Bio import SeqIO
from tqdm import tqdm

sys.path.insert(0, str(base_dir))     # чтобы spawn-воркеры нашли модуль zdna_worker
import zdna_worker as zw
importlib.reload(zw)

def merge_intervals(intervals):
    """Слить пересекающиеся/смежные (start, end) — убирает дубли из перекрытия кусков."""
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

# 1) кэш: проверяем/сохраняем параметры, влияющие на результат (intervals)
import json, re

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# НЕ включаем BATCH_SIZE (не меняет результат), N_PROC/THREADS_PER (только скорость),
# MAX_BP (содержимое куска определяется start/end, которые в id файла кэша)
gp = Path(FASTA_IN); _st = gp.stat()
current_params = {
    "model_dir": str(MODEL_DIR),
    "genome": str(gp), "genome_size": _st.st_size, "genome_mtime": int(_st.st_mtime),
    "CHUNK": CHUNK, "OVERLAP": OVERLAP,
    "THRESHOLD": THRESHOLD, "MIN_LEN": MIN_LEN,
}
pfile = CACHE_DIR / "params.json"
if pfile.exists():
    saved = json.loads(pfile.read_text())
    diff = {k: (saved.get(k), v) for k, v in current_params.items() if saved.get(k) != v}
    if diff:
        raise ValueError(f"Параметры кэша {CACHE_DIR} не совпадают с текущими: {diff}. "
                         "Очисти папку кэша или укажи другую CACHE_DIR.")
    print("кэш: параметры совпали ->", pfile)
else:
    pfile.write_text(json.dumps(current_params, indent=2))
    print("кэш: параметры сохранены ->", pfile)

# 2) нарезаем задачи: куски по CHUNK bp (+OVERLAP для контекста), охват ограничен MAX_BP.
#    id куска = {chrom}_{start}_{end} -> кэш-хит только при точно том же срезе.
def _safe(name):
    return re.sub(r"[^\w.-]", "_", str(name))

tasks = []
for rec in SeqIO.parse(str(FASTA_IN), "fasta"):
    if CHROMS is not None and rec.id not in CHROMS:
        continue
    seq = str(rec.seq)
    L = len(seq) if MAX_BP is None else min(len(seq), MAX_BP)
    for st in range(0, L, CHUNK):
        en = min(st + CHUNK + OVERLAP, L)
        cache_path = str(CACHE_DIR / f"{_safe(rec.id)}_{st}_{en}.tsv")
        tasks.append((rec.id, st, seq[st:en], THRESHOLD, MIN_LEN, BATCH_SIZE, cache_path))

cached = [t for t in tasks if os.path.exists(t[-1])]
todo   = [t for t in tasks if not os.path.exists(t[-1])]
print(f"кусков: {len(tasks)} | из кэша: {len(cached)} | считать: {len(todo)} "
      f"| параллель: {N_PROC} x {THREADS_PER}")

кэш: параметры совпали -> /home/bulat_kharisov/inf/results/cache/zdnabert/params.json
кусков: 12 | из кэша: 6 | считать: 6 | параллель: 6 x 16


In [7]:
# 3) инференс. spawn: свежий интерпретатор, не наследует torch из ядра -> без дедлока
results = []

# кэш-хиты читаем в родителе — без пула и без загрузки модели
for t in tqdm(cached, total=len(cached)):
    results.extend(zw.process_chunk(t))

# остальное считаем параллельно (пул создаём только если есть что считать)
if todo:
    ctx = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=N_PROC, mp_context=ctx,
                             initializer=zw.init_worker,
                             initargs=(str(MODEL_DIR), THREADS_PER)) as ex:
        futures = [ex.submit(zw.process_chunk, t) for t in todo]
        for f in tqdm(as_completed(futures), total=len(futures), desc="куски"):
            results.extend(f.result())
else:
    print("всё из кэша — инференс не нужен")

куски: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [01:37<00:00, 16.27s/it]


In [ ]:
# 3) сливаем интервалы по хромосомам и пишем BED
by_chrom = defaultdict(list)
for chrom, s, e in results:
    by_chrom[chrom].append((s, e))

n_written = 0
with open(BED_OUT, "w") as bed:
    for chrom in sorted(by_chrom):
        for s, e in merge_intervals(by_chrom[chrom]):
            bed.write(f"{chrom}\t{s}\t{e}\tZDNABERT\t{e - s}\t.\n")
            n_written += 1
print("готово ->", BED_OUT, "| интервалов:", n_written)

готово -> /home/bulat_kharisov/inf/results/bed/zdnabert.bed | интервалов: 67


: 

## Итог и заметки

- `results/bed/zdnabert.bed` — предсказанные Z-участки (BED6: chrom, start, end, name, length, strand).
- Для отчёта зафиксируй параметры: модель (`HG kouzine`), `THRESHOLD=0.5`, `MIN_LEN=10`, `CHUNK/OVERLAP`, `N_PROC×THREADS_PER`, и какой охват (`CHROMS` + `MAX_BP`) посчитал (пилот).

**Если OOM / долго:**
- уменьши `MAX_BP` (главный рычаг) или `CHUNK`;
- держи `N_PROC*THREADS_PER ≤ 96`; оптимум по бенчмаркам ~`8×12`/`12×8`;
- память: каждый процесс грузит свою копию модели (~340 МБ) — следи за `N_PROC`.

**Проверка эквивалентности:** батч с `attention_mask` даёт тот же результат, что поштучный инференс (проверено: max abs diff ~1e-11). Окна/куски с `OVERLAP` + слияние интервалов не теряют Z-сайты на стыках.

Дальше по плану: то же по zhunt и G-квадруплексам → ещё 2 BED, затем разметка участков и таблицы (пункты 4-6).